In [1]:
from pathlib import Path
import json
import yaml

from scipy.spatial.distance import cdist
import numpy as np 
import pandas as pd
import pymeshlab
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plyfile import PlyData
from sklearn.neighbors import NearestNeighbors

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Pheno4D

In [ ]:
data_folder = Path('../data/Pheno4D/raw/Maize01')
file_paths = sorted(data_folder.glob('*'))

In [ ]:
df = pd.read_csv(file_paths[0], sep=' ', header=None, index_col=False)

In [ ]:
assert len(df.columns) >= 3, f"Expected at least 3 columns, got {len(df.columns)}"

In [ ]:
data = np.loadtxt(file_paths[0])
data

In [ ]:
SCALAR_DIM = 4
points = data[:, :3]
labels = data[:, SCALAR_DIM] 

In [ ]:
unique_labels = np.unique(labels)
color_map = plt.get_cmap("tab20")  # or any other matplotlib colormap
label_to_color = {label: color_map(i / len(unique_labels))[:3] for i, label in enumerate(unique_labels)}
colors = np.array([label_to_color[label] for label in labels])

In [ ]:
fraction = 0.1
N = points.shape[0]
sample_indices = np.random.choice(N, size=int(N * fraction), replace=False)

In [ ]:
labels

### Get one leaf

In [ ]:
sample_indices = np.where(points[:, 2] > -5)[0]
sample_indices = np.where(labels[:] == 1.)
sample_indices

In [ ]:
points_downsampled = points[sample_indices]
labels_downdownsampled = labels[sample_indices]
colors_downsampled = colors[sample_indices]

In [ ]:
colors.shape

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points_downsampled[:, 0], points_downsampled[:, 1], points_downsampled[:, 2], c=colors_downsampled, s=1)  # s=1 for small point size

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Point Cloud Colored by Column')

plt.show()

In [ ]:
plt.hist(points[:, 2], bins=100)
plt.xlim(-10, 1)

In [ ]:
np.median(points[:, 2])

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=points_downsampled[:, 0],
    y=points_downsampled[:, 1],
    z=points_downsampled[:, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=colors_downsampled,  # Can use labels or other scalar for color
        colorscale='Viridis',
        opacity=0.8
    )
)])
fig.show()

In [ ]:
# # Create Open3D point cloud
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(points_downsampled)

# # If you have color data (float RGB values between 0 and 1)
# pcd.colors = o3d.utility.Vector3dVector(colors_downsampled)

# # Save to PLY
# o3d.io.write_point_cloud("../downsampled_point_cloud.ply", pcd)

## BonnBeetClouds3D

In [7]:
base_path = Path('~/public-mnt/Evaluation/Projects/KP0034_jgajardo/data/external/BonnBeetClouds3Dv0').expanduser().absolute()
base_path.exists()

True

### Check PLY files

In [8]:
data_folder = base_path / 'val'
sample_file = list(data_folder.glob('*'))[0]

In [9]:
open3d_pcd = o3d.io.read_point_cloud(str(sample_file))
pcd_np = np.asarray(open3d_pcd.points)

Open3D doesn't support reading scalar fields from PLY files, so we'll use `plyfile` instead.

Iterate over folder to check the amount of scalar fields (look for leaf tip)

In [10]:
def get_ply_attributes(ply_file_path):
    ply_data = PlyData.read(ply_file_path)
    vertex = ply_data['vertex']
    return vertex.properties

In [11]:
get_ply_attributes(sample_file)

(PlyProperty('x', 'float'),
 PlyProperty('y', 'float'),
 PlyProperty('z', 'float'),
 PlyProperty('nx', 'float'),
 PlyProperty('ny', 'float'),
 PlyProperty('nz', 'float'),
 PlyProperty('red', 'uchar'),
 PlyProperty('green', 'uchar'),
 PlyProperty('blue', 'uchar'),
 PlyProperty('plant_ids', 'int'),
 PlyProperty('confidence', 'uchar'),
 PlyProperty('leaf_ids', 'int'),
 PlyProperty('plot', 'double'))

In [12]:
for file in data_folder.glob('*'):
    print(len(get_ply_attributes(sample_file)))

13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13


There is no keypoint_id in the data as advertised in the paper :(

### Check traits

In [13]:
def load_yaml(yaml_file_path):
    """Load YAML file and return its contents as a Python object."""
    with open(yaml_file_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [14]:
traits_file_path = base_path / "phenotypic_trait_gt.yaml"
traits = load_yaml(traits_file_path)
len(traits.keys())

57

In [15]:
def get_all_ply_files(base_path: Path):
    """Get all PLY files organized by dataset split (train/val/test)."""
    ply_files = {}
    
    for split in ["train", "val", "test"]:
        split_dir = base_path / split
        if split_dir.exists():
            ply_files[split] = list(split_dir.glob("*.ply"))
        else:
            ply_files[split] = []
    
    return ply_files


def extract_patch_id_from_filename(ply_path: Path):
    """Extract patch ID from PLY filename (e.g., 'patch_0_1.ply' -> '0_1')."""
    filename = ply_path.stem
    if filename.startswith("patch_"):
        return filename[6:]  # Remove 'patch_' prefix
    return None


def find_ply_files_for_traits(
    traits: dict, base_path: Path
):
    """
    Find PLY files that match trait keys.
    
    Args:
        traits: Dictionary with trait data where keys are patch IDs (e.g., '0_1', '0_10')
        base_path: Path to the dataset directory
    
    Returns:
        Dictionary mapping trait keys to their corresponding PLY file paths by split.
        Format: {trait_key: {'train': path_or_none, 'val': path_or_none, 'test': path_or_none}}
    """
    # Get all PLY files organized by split
    ply_files_by_split = get_all_ply_files(base_path)
    
    # Create mapping from patch ID to file path for each split
    patch_to_path = {}
    for split, files in ply_files_by_split.items():
        patch_to_path[split] = {}
        for ply_path in files:
            patch_id = extract_patch_id_from_filename(ply_path)
            if patch_id:
                patch_to_path[split][patch_id] = ply_path
    
    # Match trait keys with PLY files
    trait_to_ply = {}
    for trait_key in traits.keys():
        trait_to_ply[trait_key] = {}
        for split in ["train", "val", "test"]:
            trait_to_ply[trait_key][split] = patch_to_path[split].get(trait_key, None)
    
    return trait_to_ply


def get_trait_ply_pairs(
    traits: dict, base_path: Path
):
    """
    Get list of (trait_key, ply_path, trait_data) tuples for matched files.
    
    Returns only pairs where a PLY file exists for the trait.
    """
    trait_to_ply = find_ply_files_for_traits(traits, base_path)
    
    pairs = []
    for trait_key, trait_data in traits.items():
        for split in ["train", "val", "test"]:
            ply_path = trait_to_ply[trait_key][split]
            if ply_path is not None:
                pairs.append((trait_key, ply_path, trait_data))
    
    return pairs

In [16]:
print("Number of traits:", len(traits))
print("First few trait keys:", list(traits.keys())[:5])

# Test PLY file matching functions
print("\n=== PLY File Matching ===")
trait_to_ply = find_ply_files_for_traits(traits, base_path)

# Count matches by split
matches_by_split = {"train": 0, "val": 0, "test": 0}
for trait_key, split_paths in trait_to_ply.items():
    for split, ply_path in split_paths.items():
        if ply_path is not None:
            matches_by_split[split] += 1

print("Matches by split:")
for split, count in matches_by_split.items():
    print(f"  {split}: {count}")

# Get trait-PLY pairs
pairs = get_trait_ply_pairs(traits, base_path)
print(f"\nTotal trait-PLY pairs: {len(pairs)}")

# Show some examples
print("\nFirst few trait-PLY pairs:")
for trait_key, ply_path, trait_data in pairs[:3]:
    print(f"  {trait_key}: {ply_path.name}")
    print(f"    traits: {list(trait_data.keys())}")

# Check for traits without PLY files
missing_files = []
for trait_key, split_paths in trait_to_ply.items():
    if all(path is None for path in split_paths.values()):
        missing_files.append(trait_key)

if missing_files:
    print(f"\nTraits without PLY files: {len(missing_files)}")
    print(f"Examples: {missing_files[:5]}")
else:
    print("\nAll traits have corresponding PLY files!")

Number of traits: 57
First few trait keys: ['0_1', '0_10', '0_11', '0_12', '0_13']

=== PLY File Matching ===
Matches by split:
  train: 3
  val: 0
  test: 0

Total trait-PLY pairs: 3

First few trait-PLY pairs:
  0_1: patch_0_1.ply
    traits: ['blade_len', 'blade_wid', 'leaf_len', 'stem_len', 'stem_wid']
  0_18: patch_0_18.ply
    traits: ['blade_len', 'blade_wid', 'leaf_len', 'stem_len', 'stem_wid']
  1_1: patch_1_1.ply
    traits: ['blade_len', 'blade_wid', 'leaf_len', 'stem_len', 'stem_wid']

Traits without PLY files: 54
Examples: ['0_10', '0_11', '0_12', '0_13', '0_14']


## TrackPlant3D

In [2]:
# Helper functions
def load_point_cloud(file_path):
    data = np.loadtxt(file_path)
    return data[:, :3], data[:, 3:]  # Assuming last columns are colors

def plot_interactive_with_keypoint(points, keypoint_indices: list):
    """
    Interactive plot with keypoint highlighted
    """
    colors = ['gray'] * len(points)
    sizes = [2] * len(points)

    # Highlight keypoints
    for i, idx in enumerate(keypoint_indices):
        colors[idx] = 'red'
        sizes[idx] = 8
    
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=sizes,
            color=colors,
            opacity=0.8
        ),
        text=[f'Index: {idx}' for idx in range(points.shape[0])],  # Hover info
        hovertemplate='Index: %{text}<br>X: %{x}<br>Y: %{y}<br>Z: %{z}<extra></extra>'
    )])
    
    fig.update_layout(
        title=f'Point Cloud with Keypoint (Indices: {keypoint_indices})',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        )
    )
    fig.show()

def load_sphere_data(json_path):
    """Load sphere data from labelCloud format JSON file"""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    spheres = []
    for sphere in data.get('spheres', []):
        center = sphere['center']
        spheres.append({
            'x': center['x'],
            'y': center['y'], 
            'z': center['z'],
            'radius': sphere['radius'],
            'name': sphere['name']
        })
    return spheres, data.get('filename', '')

def find_closest_sphere(sphere_list, query_sphere):
    """
    Find the closest sphere in sphere_list to query_sphere
    using Euclidean distance.
    Args:
        query_sphere: dict with 'x', 'y', 'z' keys
        sphere_list: list of sphere dictionaries
    
    Returns:
        closest_sphere: dict - the closest sphere
        min_distance: float - distance to closest sphere
        closest_index: int - index of closest sphere in the list
    """
    if not sphere_list:
        return None, float('inf'), -1

    query_coords = np.array([query_sphere['x'], query_sphere['y'], query_sphere['z']])
    sphere_coords = np.array([[s['x'], s['y'], s['z']] for s in sphere_list])
    
    # Calculate distances to all spheres
    distances = np.linalg.norm(sphere_coords - query_coords, axis=1)

    # Find closest
    closest_index = np.argmin(distances)
    min_distance = distances[closest_index]
    closest_sphere = sphere_list[closest_index]
    
    return closest_sphere, min_distance, closest_index


def sort_names(paths: list [Path]) -> list[Path]:
    return sorted(paths, key=lambda x: int(x.stem.split('_')[0]))

def get_mean_distance_between_spheres(pred_tips_paths, manual_tips_paths):
    """
    Calculate the mean distance between spheres in Open3D tips and manual tips.
    
    Args:
        open3d_tips_paths: Path to Open3D tips directory
        manual_tips_paths: Path to manual tips directory
    
    Returns:
        mean_distance: float - average distance between matched spheres
        """
    distances = []
    for pred_json, gt_json in zip(sort_names(pred_tips_paths.glob('*.json')), sort_names(manual_tips_paths.glob('*.json'))):
        assert pred_json.stem == gt_json.stem, f"Files {pred_json.name} and {gt_json.name} do not match!"

        # Load sphere data
        pred_spheres, _ = load_sphere_data(pred_json)
        gt_spheres, _ = load_sphere_data(gt_json)

        for query_sphere in pred_spheres:
            # We don't have the leaf_id in the jsons so we just find the closest sphere
            # in the ground truth. This should yield the corresponding sphere in almost all cases and fine for a quick evaluation
            _, min_distance, _ = find_closest_sphere(
                    sphere_list=gt_spheres,
                    query_sphere=query_sphere
                )
            distances.append(min_distance)

    return np.average(distances)

### EDA

In [ ]:
crop = 'tomato'
data_folder = Path(f'../data/TrackPlant3D/gt/{crop}')
file_paths = sorted(data_folder.glob('*'), key=lambda x: int(x.stem.split('_')[0]))

In [4]:
data, labels = load_point_cloud(file_paths[0])
data.shape, labels.shape

((8892, 3), (8892, 1))

In [7]:
number_of_leaves = []
for file_path in file_paths:
    labels = load_point_cloud(file_path)[1]
    #print(f"{file_path.name}: {np.unique(labels)}")
    number_of_leaves.append(len(np.unique(labels)) - 1)
print(f"Total number of leaves in {crop}: {np.sum(number_of_leaves)}")

Total number of leaves in maize: 86


In [17]:
def get_total_leaf_count(crop):
    data_folder = Path(f'../data/TrackPlant3D/gt_corrected_v1/{crop}')
    file_paths = sorted(data_folder.glob('*'), key=lambda x: int(x.stem.split('_')[0]))
    number_of_leaves = []
    for file_path in file_paths:
        labels = load_point_cloud(file_path)[1]
        number_of_leaves.append(len(np.unique(labels)) - 1)
    return int(np.sum(number_of_leaves))

counts = [get_total_leaf_count(crop) for crop in ['maize', 'sorghum', 'tobacco', 'tomato']]
print(counts)
print('Total: ', sum(counts))

[86, 262, 297, 930]
Total:  1575


### Check leaf tips labels/arrays (output of LabelCloud)

In [ ]:
bin_files = sorted(Path('../data/TrackPlant3D/keypoints/sorghum/arrays').glob('*.bin'), key=lambda x: int(x.stem.split('_')[0]))
path = bin_files[-1]
print(path)
labels = np.fromfile(path, dtype=np.int8)
print(labels.shape)
np.unique(labels, return_counts=True)

In [ ]:
np.unique(labels, return_counts=True)

### Export single leaf for meshing

In [ ]:
crop = 'tomato'
data_folder = Path(f'../data/TrackPlant3D/gt/{crop}')
file_paths = sorted(data_folder.glob('*'), key=lambda x: int(x.stem.split('_')[0]))

In [ ]:
file_path = file_paths[0]
points, labels = load_point_cloud(file_path)
file_path.name, points.shape, labels.shape

In [ ]:
unique_labels = np.unique(labels)
unique_labels

In [ ]:
leaf_id = 1.0 # start from 1.0 for the first leaf
assert leaf_id in unique_labels, f"Leaf ID {leaf_id} not found in unique labels: {unique_labels}"
assert leaf_id != 0.0, "Leaf ID should not be 0.0, which represents the stem."

sample_indices = np.where(labels[:] == unique_labels[int(leaf_id)])[0]
leaf_points = points[sample_indices]
leaf_points.shape

In [ ]:
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(leaf_points)
# o3d.io.write_point_cloud(f"../{file_path.stem}_leaf{int(leaf_id)}.ply", pcd)

Plot keypoint

In [ ]:
index = 5311  # Just for checking, found with Meshlab -> meshing with ball pivot -> Colorize by geodesic distance -> The info to check point index
plot_interactive_with_keypoint(leaf_points, keypoint_index=index)

**Goal:** find keypoint automatically with pymeshlab!

### Explore automatic keypoint detection

In [ ]:
# Mesh the leaf to enable geodesic distance computation
ms = pymeshlab.MeshSet()
mesh = pymeshlab.Mesh(vertex_matrix=leaf_points)
ms.add_mesh(mesh, f"leaf_{int(leaf_id)}")
ms.generate_surface_reconstruction_ball_pivoting()

In [ ]:
centroid = np.mean(leaf_points, axis=0)
leaf_centroid_index = np.argmin(np.linalg.norm(leaf_points - centroid, axis=1)) # closest data point to the centroid
leaf_centroid = np.array(leaf_points[leaf_centroid_index], dtype=np.float32)

In [ ]:
# Plot leaf centroid
plot_interactive_with_keypoint(leaf_points, keypoint_index=leaf_centroid_index)

In [ ]:
ms.compute_scalar_by_geodesic_distance_from_given_point_per_vertex(startpoint=leaf_centroid)
geodist = ms.current_mesh().vertex_scalar_array()
geodist.shape

In [ ]:
tip_index = np.argmax(geodist)
tip_index

In [ ]:
# Plot farthest point from centroid (either tip or insertion point)
plot_interactive_with_keypoint(leaf_points, keypoint_index=tip_index)

But how to know if farthest point from leaf centroid is the tip and insertion point? use stem points!

In [ ]:
stem_indices = np.where(labels == 0)[0]
stem_points = points[stem_indices]
stem_points

In [ ]:
leaf_stem_points = np.vstack((leaf_points, stem_points))

In [ ]:
plot_interactive_with_keypoint(leaf_stem_points, keypoint_index=leaf_centroid_index)

In [ ]:
# Find the closest geodesic point with stem label (in first `len(stem_points)` entries)
ms = pymeshlab.MeshSet()
mesh = pymeshlab.Mesh(vertex_matrix=leaf_stem_points)
ms.add_mesh(mesh, f"leaf_stem_{leaf_id}")
ms.generate_surface_reconstruction_ball_pivoting() # Meshing leaf and stem together doesn't work well though

Another approach is to go first get farthest point from leaf centroid (yielding keypoint A), then get the farthest point from keypoint A along th leaf (yielding keypoint B), then we can find which on is the insertion point by checking which one is closer to a stem point

In [ ]:
# This seems to work nicely!
ms.add_mesh(pymeshlab.Mesh(vertex_matrix=leaf_points), f"leaf_{leaf_id}")
ms.generate_surface_reconstruction_ball_pivoting()

centroid = np.mean(leaf_points, axis=0)
first_index = np.argmin(np.linalg.norm(leaf_points - centroid, axis=1))
first_point = np.array(leaf_points[first_index], dtype=np.float32)

# First get farthest point from centroid (could be tip or insertion point)
ms.compute_scalar_by_geodesic_distance_from_given_point_per_vertex(startpoint=first_point)
geodist1 = ms.current_mesh().vertex_scalar_array()
index_A = np.argmax(geodist1)
point_A = leaf_points[index_A]

# Now get farthest point from point_A
ms.compute_scalar_by_geodesic_distance_from_given_point_per_vertex(startpoint=np.array(point_A, dtype=np.float32))
geodist2 = ms.current_mesh().vertex_scalar_array()
index_B = np.argmax(geodist2)
point_B = leaf_points[index_B]

In [ ]:
keypoint_index = index_A # change to index_B to plot the other point
plot_interactive_with_keypoint(leaf_points, keypoint_index=keypoint_index)

In [ ]:
np.linalg.norm(stem_points - point_A, axis=1).min() # this is smaller than for B so this must be insertion point! True

In [ ]:
np.linalg.norm(stem_points - point_B, axis=1).min()

Point A must be insertion point and point B the tip, because the it has a smaller minimum distance to the stem

More concisely:

In [ ]:
# Get distances and determine insertion point
distances = [np.linalg.norm(stem_points - point_A, axis=1).min(),
            np.linalg.norm(stem_points - point_B, axis=1).min()]
points = [point_A, point_B]
indices = [index_A, index_B]

insertion_idx = np.argmin(distances)
tip_idx = 1 - insertion_idx  # The other point

insertion_point = points[insertion_idx]
insertion_index = indices[insertion_idx]
tip_point = points[tip_idx]
tip_index = indices[tip_idx]

In [ ]:
plot_interactive_with_keypoint(leaf_points, keypoint_index=insertion_index)

### Evaluate automatic leaf tip detection

Evaluate leaf tip found by pymeshlab / open3D pipeline (running [`detect_leaf_tips_trackplant3D.py`](../src/data_preprocessing/detect_leaf_tips_trackplant3D.py)) with `ball_pivoting` meshing, compared to manually labeled leaf tips for maize plants. **Note that we compare to manually annotated leaf tips in labelCloud as spheres centers, which do not necessarily coincide with a true data point in the point cloud.** In any case this should provide a good estimate of which pipeline is better.

In [ ]:
crop = 'maize'
method = 'ball_pivoting'
manual_tips_paths = Path(f'../data/TrackPlant3D/keypoints/{crop}/jsons')

In [ ]:
folder_name = f'keypoints_autolabel'
open3d_tips_paths = Path(f'../data/TrackPlant3D/{folder_name}/{method}_open3d/{crop}/leaf_tip_spheres')
pymeshlab_tips_paths = Path(f'../data/TrackPlant3D/{folder_name}/{method}_pymeshlab/{crop}/leaf_tip_spheres')
assert all([open3d_tips_paths.exists(), pymeshlab_tips_paths.exists(), manual_tips_paths.exists()])

In [ ]:
print("L2 error Open3D vs Manual:", get_mean_distance_between_spheres(open3d_tips_paths, manual_tips_paths))
print("L2 error PyMeshLab vs Manual:", get_mean_distance_between_spheres(pymeshlab_tips_paths, manual_tips_paths))

Pipeline in meshlab seems to be more accurate than the one in open3D, so we can use that for the other crops!

In [ ]:
folder_name = f'keypoints_autolabel_median_centroid'
open3d_tips_paths = Path(f'../data/TrackPlant3D/{folder_name}/{method}_open3d/{crop}/leaf_tip_spheres')
pymeshlab_tips_paths = Path(f'../data/TrackPlant3D/{folder_name}/{method}_pymeshlab/{crop}/leaf_tip_spheres')
assert all([open3d_tips_paths.exists(), pymeshlab_tips_paths.exists(), manual_tips_paths.exists()])

In [ ]:
print("L2 error Open3D vs Manual:", get_mean_distance_between_spheres(open3d_tips_paths, manual_tips_paths))
print("L2 error PyMeshLab vs Manual:", get_mean_distance_between_spheres(pymeshlab_tips_paths, manual_tips_paths))

Also noticed that using the median for the leaf centroid is better than the mean, as it is less sensitive to outliers

### Visually inspect leaf tips

In [ ]:
crop = 'maize'
point_cloud_paths = sort_names(Path(f'../data/TrackPlant3D/gt/{crop}').glob('*.txt'))
leaf_tips_paths = sort_names(Path(f'../data/TrackPlant3D/keypoints_autolabel/ball_pivoting_pymeshlab/{crop}/leaf_tips').glob('*.json'))

In [ ]:
for point_cloud_path, leaf_tips_path in zip(point_cloud_paths, leaf_tips_paths):
    print(f"Processing {point_cloud_path.name} and {leaf_tips_path.name}")
    assert point_cloud_path.stem == leaf_tips_path.stem, "File names dont match!"

    # Load point cloud
    points, labels = load_point_cloud(point_cloud_path)
    
    # Load leaf tips
    with open(leaf_tips_path, 'r') as f:
        leaf_tips = json.load(f)
    
    leaf_tip_indices = [leaf_tip['tip_global_index'] for leaf_tip in leaf_tips.values() if 'tip_global_index' in leaf_tip]
    plot_interactive_with_keypoint(points, keypoint_indices=leaf_tip_indices)
    break


### Inpsect points in plant

In [17]:
crop = 'tobacco'
filepath = sort_names(Path(f'../data/TrackPlant3D/gt/{crop}').glob('*.txt'))[210]
print(filepath.name)
points, labels = load_point_cloud(filepath)
print(f"Number of leaves:", len(np.unique(labels)) - 1)
print(np.unique(labels, return_counts=True))

# Load keypoints if they exist
keypoints_path = Path(f'~/public-mnt/Evaluation/Projects/KP0034_jgajardo/projects/2024-discrete4DGS-paper/data/TrackPlant3D/keypoints/leaf_tips/{crop}/{filepath.name}').expanduser().absolute()
data = None
if keypoints_path.exists():
    data = np.loadtxt(keypoints_path, delimiter=',')

plot_interactive_with_keypoint(points, keypoint_indices=data[..., 0].astype(int).flatten().tolist() if data is not None else [])

IndexError: list index out of range

In [16]:
plot_interactive_with_keypoint(points, keypoint_indices=[1,2,3,4])